# Decision-aware sampling: overview
> The design, its rationale and the approach so far, on the `scoring-experiments` branch

This page summarises the work and gives a reading order. The details are in `scoring-design.md` (the proposal), `scoring-plan.md` (the steps and their results), `nbs/07_scoring.ipynb` (the code) and `nbs/_scoring_experiments.ipynb` (the comparisons). Nothing here changes trufl's published API yet.

## Reading order

0. A primer on Bayesian statistics: what samples say about a unit's share above the level.
1. A sampling campaign, round by round: `Campaign` and strategies.
2. The MCDM layer: how trufl ranks units today.
3. What a good campaign means: scoring estimates against scoring decisions.
4. The sawtooth in the restriction rule: why one more sample can make a decision less reliable.
5. From `bayes` to `voi`: allocating where the dashboard's decision is most likely to be wrong.
6. Spatial correlation and where to place samples: Gaussian random fields, placements, plumes.
7. Costs, dilemmas and a health limit: different costs for the two errors, and what goes wrong without a limit.

## The question

trufl allocates sampling effort after a nuclear accident. Its aim was to describe every unit well. Decision makers need something else: to restrict the right units. The DSS4NAFA dashboard restricts a unit when more than an accepted proportion (10%) of its samples is above the permissible level. A campaign is good when that rule, applied to its samples, gives the decisions the truth would give.

Scored on decisions, the ranking of strategies changes. Scored on how well unit means are estimated, no strategy beat equal allocation. Scored on wrong restriction decisions, decision-aware strategies did much better.

## The design

Five parts, each with one job:

- A decision rule, given as input: the DSS4NAFA rule now. trufl does not copy the dashboard; it takes its rule.
- An information layer: `voi` gives each sample to the unit where it most lowers the expected cost of the rule's decision. It predicts what the rule will decide from the samples, using a beta posterior on each unit's share, so it sees the sawtooth and uses a prior (airborne survey or dispersion model) correctly.
- Placement within a unit: `spread` places each new sample away from the unit's earlier samples. With spatially correlated contamination, spread samples tell more than random ones.
- A value layer: MCDM sets per-unit costs from value criteria (crop value, food security, population, self-consumption). A needless restriction and a missed one get separate costs, and one ratio `r` weighs them against each other.
- A health limit: a bound on each unit's probability of a missed restriction, met before the costs apply.

A simulation module (`Campaign`, strategies, scores, synthetic ground truths) compares strategies before an emergency.

## The rationale

- MCDM's role changes. Information criteria (standard deviation, Moran's I) were proxies for uncertainty; `voi` handles uncertainty directly. MCDM keeps what it is good at: making the stakes explicit and explainable to decision makers.
- The rule decides, not trufl. Costs and bounds choose where to sample, never what to decide. Changing the decision rule for valuable crops would trade health for money, which is a policy choice.
- Health is a limit, not a cost. Under the dashboard's rule, more samples can only restrict an unrestricted unit. With costs alone, `voi` stops sampling clean-looking units where a restriction is expensive, and contamination there goes unfound. The bound prevents this.
- Synthetic ground truths for publication. The Fukushima data are for internal checks only.

## What the experiments showed

Wrong restriction decisions after six rounds of 150 samples on 100 units, with the dashboard's rule (restrict from 10% of samples above the level):

- `bayes` (allocation by posterior uncertainty) beats equal allocation: 7.8 against 10.0 on `ground-truth-02`, and in five of six settings on Fukushima windows.
- A prior makes `bayes` worse, because the dashboard ignores the prior when it decides. `voi`, which predicts the dashboard's decision, uses a prior well: 4.3 with a perfect prior on `ground-truth-02`.
- `voi` needed to look 10 samples ahead, not 5. With 5, units with an early exceedance stayed restricted needlessly. With 10, `voi` without a prior matches `bayes`, and with a good prior it does better; a poor prior can still make it worse (on `ground-truth-02`, the survey prior).
- On ten synthetic plumes: a survey prior lowers wrong decisions by 0.8 per campaign and a dispersion-model prior by 0.5. Spread placement shows no clear gain (about 0.4, in 5 of 10 fields). With spread placement and a survey prior, `voi` makes 2.7 wrong decisions, the fewest of the realistic strategies.
- `voi` and `bayes` make about as many errors but split them differently: `voi` more needless restrictions, `bayes` more missed ones. The ratio `r` makes that split a setting.
- Costs alone weaken health protection where value is high: with a needless restriction costing 5 in half the units, missed restrictions there rose from 1.3 to 8.6 with the dispersion prior. A bound of 20% brought them to 1.7, and a bound of 5% to 0.2, but a strict bound leaves no room for the costs.

## The approach

Small steps, each answering one question and recorded in `scoring-plan.md`. New work stays in `nbs/07_scoring.ipynb`, which is not exported. Results are checked on several ground truths, and findings from one field are replicated over ten before anything builds on them. The published modules change only after your decisions.

## Open decisions

For you:

- The library changes listed in `scoring-design.md`: `StdCB`, `MoranICB`, `LINEAR1`, `rank_to_sample`, and the names and modules of the new public API.
- Costs per unit or per crop, and how MCDM criteria become the two costs.

For DSS4NAFA:

- The sawtooth in the rule: one more sample can make a decision less reliable.
- Whether decisions should use prior information, such as the airborne survey.
- Whether decision makers would set `r` openly, and what bound on missed restrictions is acceptable.

Not yet done: measurement noise (#11), several foodstuffs sharing one budget, "partially restrict", and an effective sample count for spread placement.